In [5]:
import pandas as pd
from sklearn.feature_extraction.text import HashingVectorizer, TfidfTransformer
from sklearn.linear_model import SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

In [23]:
quartered_df = pd.read_csv("C:\\Users\\User\\Desktop\\lenta_ru_news_2019_2023.csv")

In [25]:
lenta_topics = {
    "Россия": 0,
    "Экономика": 1,
    "Силовые структуры": 2,
    "Бывший СССР": 3,
    "Спорт": 4,
    "Забота о себе": 5,
    "Здоровье": 5,
    "Строительство": 6,
    "Путешествия": 7,
    "Наука и техника": 8,
    "Интернет и СМИ": 0,
    "Бизнес": 1,
    }

quartered_df["number"] = quartered_df["topic"].apply(lambda x: lenta_topics[x] if x in lenta_topics else None)
quartered_df = quartered_df.dropna(subset=['number'])
quartered_df = quartered_df.dropna(subset=['text'])

quartered_df["number"] = quartered_df["number"].apply(lambda x: int(x))

In [26]:
quartered_df.head(5)

,url,title,text,topic,tags,date,number
0,https://lenta.ru/news/2019/12/15/prsm/,Россиянам дали советы по выбору чая,Россиянам дали советы при выборе чая. Рекоменд...,Россия,Общество,2019-12-15,0
1,https://lenta.ru/news/2019/12/15/fb/,В Госдуме назвали японское заявление о Курилах...,Спикер Госдумы Вячеслав Володин назвал угрозой...,Россия,Политика,2019-12-15,0
3,https://lenta.ru/news/2019/12/15/alba/,Полицейские застрелили порезавшего мать буйног...,В Москве полицейские застрелили мужчину при по...,Силовые структуры,Криминал,2019-12-15,2
5,https://lenta.ru/news/2019/12/15/apple/,Свадьба трансгендеров в России попала на видео,В ЗАГСе Казани официально вступила в брак пара...,Россия,Общество,2019-12-15,0
6,https://lenta.ru/news/2019/12/15/tattoo/,Россиян предостерегли от нового способа кражи ...,Россиян предостерегли от нового способа кражи ...,Россия,Общество,2019-12-15,0


In [27]:
with open("russian.txt", "r", encoding="utf-8") as file:
    stop_words = [line.strip() for line in file]

In [28]:
# 1. Разбиение на обучающую и тестовую выборки
x_train, x_test, y_train, y_test = train_test_split(
    quartered_df.text,
    quartered_df.number,
    test_size=0.25,
    random_state=13
)

In [29]:
# 2. Строим Pipeline: Hashing → TF-IDF → LinearSVC
pipeline = Pipeline([
    # HashingVectorizer: фиксированная размерность, нет словаря
    ('hash', HashingVectorizer(
        n_features=2**18,       # мощность хеш-пространства (~260k признаков)
        alternate_sign=False,   # чтобы TF-IDF корректно считал частоты
        norm=None,              # нормировку сделает TF-IDF
        lowercase=True,         # приводить к нижнему регистру
        token_pattern=r"(?u)\b\w+\b"  # простой шаблон токенов
    )),
    # преобразуем «сырые» счётчики в TF-IDF
    ('tfidf', TfidfTransformer()),
    # линейный SVM с балансировкой классов
    ('clf', LinearSVC(
        class_weight='balanced', 
        max_iter=10000, 
        random_state=13
    )),
])

In [30]:
# 3. Обучение
pipeline.fit(x_train, y_train)

Pipeline(steps=[('hash',
                 HashingVectorizer(alternate_sign=False, n_features=262144,
                                   norm=None, token_pattern='(?u)\\b\\w+\\b')),
                ('tfidf', TfidfTransformer()),
                ('clf',
                 LinearSVC(class_weight='balanced', max_iter=10000,
                           random_state=13))])

In [31]:
# 4. Предсказание и оценка
y_pred = pipeline.predict(x_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.93      0.89      0.91     29468
           1       0.92      0.94      0.93     12283
           2       0.91      0.93      0.92      7726
           3       0.89      0.92      0.91     12852
           4       0.98      0.99      0.99      7030
           5       0.88      0.93      0.90      1861
           7       0.94      0.95      0.95      4788
           8       0.93      0.94      0.93      6327

    accuracy                           0.92     82335
   macro avg       0.92      0.94      0.93     82335
weighted avg       0.92      0.92      0.92     82335

